# Stage 4: Feature Engineering — Feature Creation
**Member:** M4 (Student ID: IT004)  
**Assigned Preprocessing Technique:** Feature Engineering (Domain-Specific Feature Creation & Behavioral Derivation)  
**Dataset:** [Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) (UCI Machine Learning Repository)  
**Pipeline Position:** Stage 4 (Sequential)  
**Input:** `results/outputs/stage3_outliers_removed.csv`  
**Output:** `results/outputs/stage4_features_created.csv`

---

## 1. Explanation of the Technique

Raw dataset features often capture static snapshots in time (e.g. bill amount on September 2005). Machine learning models perform significantly better when features represent **underlying behavioral dynamics, trends, and risk ratios**.
In credit risk assessment:
- Static balances do not indicate whether a client is living on credit limits or maintaining healthy surplus.
- Behavioral ratios (e.g. credit utilization, payment-to-bill ratio, maximum delinquency) provide direct financial signals of financial strain and repayment capacity.

---

## 2. Justification for THIS Dataset Specifically

The raw dataset contains 6 monthly snapshots for bills (`BILL_AMT1` to `BILL_AMT6`), payments (`PAY_AMT1` to `PAY_AMT6`), and repayment status (`PAY_0` to `PAY_6`). We engineer domain-informed features:

1. **`avg_bill_amt`**: $\frac{1}{6} \sum_{i=1}^6 \text{BILL\_AMT}_i$
   - Smooths transient monthly expenditure noise to capture the client's average balance.
2. **`avg_pay_amt`**: $\frac{1}{6} \sum_{i=1}^6 \text{PAY\_AMT}_i$
   - Captures the client's average monthly liquidity/repayment capability.
3. **`credit_utilization`**: $\frac{\text{avg\_bill\_amt}}{\text{LIMIT\_BAL}}$
   - Classic credit bureau risk metric: high utilization indicates high credit reliance and elevated default risk. Clamped to $[0, 5]$ to handle negative bills.
4. **Monthly Payment Ratios (`payment_ratio_1` .. `payment_ratio_6`)**: $\frac{\text{PAY\_AMT}_i}{\max(\text{BILL\_AMT}_i, 1.0)}$
   - Measures whether the cardholder paid the bill in full (ratio $\ge 1$), partially ($0 < \text{ratio} < 1$), or not at all (ratio $= 0$).
5. **`avg_payment_ratio`**: Mean across the 6 monthly payment ratios.
6. **`bill_to_limit_ratio`**: $\frac{\text{BILL\_AMT1}}{\text{LIMIT\_BAL}}$ (current bill utilization).
7. **`max_delay`**: $\max(\text{PAY\_0}, \text{PAY\_2}, \dots, \text{PAY\_6})$
   - Maximum delinquent months recorded over the 6-month observation window.
8. **`delay_count`**: Count of months where payment was overdue ($\text{status} > 0$).

### Why this must be Stage 4:
- Must run **after Stage 3** so that outlier capping prevents infinite or distorted ratios from division by zero or extreme multi-million-dollar spikes.
- Must run **before Stage 5 (Scaling)** because all newly created numeric features must subsequently be scaled alongside original features.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs('results/outputs', exist_ok=True)
os.makedirs('results/eda_visualizations', exist_ok=True)

# 1. Load Stage 3 output
input_path = 'results/outputs/stage3_outliers_removed.csv'
df_s3 = pd.read_csv(input_path)
print(f"Loaded Stage 3 Data: {df_s3.shape[0]} rows, {df_s3.shape[1]} columns")
df_s3.head()


Loaded Stage 3 Data: 30000 rows, 30 columns


In [2]:
# 2. Derive Behavioral Features
df_fe = df_s3.copy()

bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
delay_cols = ['PAY_0'] + [f'PAY_{i}' for i in range(2, 7)]

# 1. Average bill and payment amounts
df_fe['avg_bill_amt'] = df_fe[bill_cols].mean(axis=1)
df_fe['avg_pay_amt'] = df_fe[pay_cols].mean(axis=1)

# 2. Credit utilization
df_fe['credit_utilization'] = (df_fe['avg_bill_amt'].clip(lower=0) / (df_fe['LIMIT_BAL'] + 1e-5)).clip(upper=5.0)

# 3. Monthly payment ratios (Payment / Bill)
for i in range(1, 7):
    bill_col = f'BILL_AMT{i}'
    pay_col = f'PAY_AMT{i}'
    denom = df_fe[bill_col].clip(lower=1.0)
    df_fe[f'payment_ratio_{i}'] = (df_fe[pay_col] / denom).clip(lower=0.0, upper=5.0)

ratio_cols = [f'payment_ratio_{i}' for i in range(1, 7)]
df_fe['avg_payment_ratio'] = df_fe[ratio_cols].mean(axis=1)

# 4. Recent bill to limit ratio
df_fe['bill_to_limit_ratio'] = (df_fe['BILL_AMT1'].clip(lower=0) / (df_fe['LIMIT_BAL'] + 1e-5)).clip(upper=5.0)

# 5. Delinquency indicators
df_fe['max_delay'] = df_fe[delay_cols].max(axis=1)
df_fe['delay_count'] = (df_fe[delay_cols] > 0).sum(axis=1)

print(f"Created 13 new behavioral features!")
print(f"Updated Dataset Shape: {df_fe.shape[0]} rows, {df_fe.shape[1]} columns")
df_fe[['avg_bill_amt', 'avg_pay_amt', 'credit_utilization', 'avg_payment_ratio', 'max_delay', 'delay_count']].head()


Created 13 new behavioral features!
Updated Dataset Shape: 30000 rows, 43 columns


In [3]:
# 3. Save Stage 4 output CSV
output_path = 'results/outputs/stage4_features_created.csv'
df_fe.to_csv(output_path, index=False)
print(f"Successfully exported Stage 4 output to: {output_path}")


Successfully exported Stage 4 output to: results/outputs/stage4_features_created.csv


In [4]:
# 4. EDA Visualization: Predictive Signal of Engineered Features
target = 'default payment next month'

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plt.subplots_adjust(wspace=0.3)

# Subplot 1: Distribution of Credit Utilization by Default Status
non_def = df_fe[df_fe[target] == 0]['credit_utilization']
defaulters = df_fe[df_fe[target] == 1]['credit_utilization']

axes[0].hist(non_def, bins=30, alpha=0.6, density=True, label='Non-Default (0)', color='#2ecc71', range=(0, 2))
axes[0].hist(defaulters, bins=30, alpha=0.6, density=True, label='Default (1)', color='#e74c3c', range=(0, 2))
axes[0].set_title('Credit Utilization Distribution by Default Class', fontweight='bold')
axes[0].set_xlabel('Credit Utilization (avg_bill_amt / LIMIT_BAL)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Subplot 2: Default Rate by Max Delay Months
delay_summary = df_fe.groupby('max_delay')[target].agg(['count', 'mean']).reset_index()
delay_summary = delay_summary[delay_summary['count'] >= 20]  # Filter small groups

axes[1].plot(delay_summary['max_delay'].astype(str), delay_summary['mean'] * 100,
             marker='o', linewidth=2.5, color='#8e44ad')
axes[1].set_title('Default Rate vs. Maximum Repayment Delay (Months)', fontweight='bold')
axes[1].set_xlabel('Max Delay (Months Overdue: <=0 Paid, >0 Overdue)')
axes[1].set_ylabel('Default Rate (%)')
axes[1].grid(True, linestyle='--', alpha=0.5)
for _, r in delay_summary.iterrows():
    axes[1].text(str(int(r['max_delay'])), r['mean'] * 100 + 2, f"{r['mean']*100:.0f}%", ha='center', fontsize=9)

plt.suptitle('M4: Feature Engineering — Predictive Power of Behavioral Features', fontsize=14, fontweight='bold')
plot_path = 'results/eda_visualizations/m4_credit_utilization_default.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"EDA plot saved to {plot_path}")


EDA plot saved to results/eda_visualizations/m4_credit_utilization_default.png


## 3. EDA Interpretation & Findings

1. **Credit Utilization Signal**:
   - Defaulters show a pronounced density shift towards high utilization (>0.80), indicating clients who have exhausted their revolving credit limit.
   - Non-defaulters exhibit lower utilization (<0.30) with substantial liquidity reserves.

2. **Maximum Delinquency Predictive Power**:
   - Clients with `max_delay <= 0` (paid duly or no delay) have a default rate under **15%**.
   - For clients with 1 month delay, default probability jumps to **36%**.
   - For clients with 2+ months delay, default probability skyrockets above **50% to 75%**!

3. **Hand-off to Stage 5 (M5)**:
   - The feature space has expanded to 43 columns including 13 domain-engineered predictors.
   - Stage 5 can now normalize and scale all continuous variables (both original and engineered) prior to selection and dimensionality reduction.
